In [1]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report


def evaluate_predictions(y_true, y_pred, model_name="Model"):
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    print(f"[{model_name}] Accuracy: {acc:.4f} | Macro-F1: {macro_f1:.4f}")
    return {"model": model_name, "accuracy": acc, "macro_f1": macro_f1}


def run_trivial_baseline(X_train, y_train, X_val, y_val, strategy="most_frequent"):
    clf = DummyClassifier(strategy=strategy, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_val)
    return evaluate_predictions(y_val, preds, model_name=f"Trivial ({strategy})"), clf


def run_simple_learned_baseline(X_train, y_train, X_val, y_val, max_iter=300):
    clf = LogisticRegression(max_iter=max_iter, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_val)
    return evaluate_predictions(y_val, preds, model_name="Simple Learned (LogReg)"), clf

In [3]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

base_dir = Path("/content/drive/MyDrive/PlantDiseaseProject/PlantVillage")
print("Base exists:", base_dir.exists())

if base_dir.exists():
    for p in sorted(base_dir.rglob("*")):
        if p.is_dir():
            subdirs = [d for d in p.iterdir() if d.is_dir()]
            if len(subdirs) >= 10:
                print("Found dataset root:", p, "| classes dirs:", len(subdirs))

Mounted at /content/drive
Base exists: True
Found dataset root: /content/drive/MyDrive/PlantDiseaseProject/PlantVillage/Plant_leave_diseases_dataset_without_augmentation | classes dirs: 39


In [4]:
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/content/drive/MyDrive/PlantDiseaseProject/PlantVillage/Plant_leave_diseases_dataset_without_augmentation")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMG_SIZE = (64, 64)
MAX_PER_CLASS = 150  # غيّرها لـ None لو عايز كل الصور (قد تأخذ وقت أطول)

X_list, y_list = [], []

for class_dir in sorted(DATA_DIR.iterdir()):
    if not class_dir.is_dir():
        continue
    class_name = class_dir.name
    imgs = [p for p in sorted(class_dir.iterdir()) if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    if MAX_PER_CLASS is not None:
        imgs = imgs[:MAX_PER_CLASS]

    for img_path in imgs:
        try:
            with Image.open(img_path) as img:
                img_resized = img.convert("RGB").resize(IMG_SIZE)
                arr = np.asarray(img_resized, dtype=np.float32).flatten() / 255.0
                X_list.append(arr)
                y_list.append(class_name)
        except Exception:
            pass

X = np.array(X_list)
y = np.array(y_list)
print(f"Loaded X shape: {X.shape}, total classes: {len(np.unique(y))}")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

trivial_res, trivial_clf = run_trivial_baseline(X_train, y_train, X_val, y_val, strategy="most_frequent")
logreg_res, logreg_clf = run_simple_learned_baseline(X_train, y_train, X_val, y_val, max_iter=300)

Loaded X shape: (5850, 12288), total classes: 39
[Trivial (most_frequent)] Accuracy: 0.0256 | Macro-F1: 0.0013


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[Simple Learned (LogReg)] Accuracy: 0.5709 | Macro-F1: 0.5669
